<div style="background: linear-gradient(90deg, #e0eafc 0%, #cfdef3 100%); border-radius: 12px; padding: 32px 24px; margin-bottom: 24px; box-shadow: 0 2px 8px rgba(0,0,0,0.07);">

# 🌟 Agentes con Memoria Persistente en Lakebase

Bienvenido a este notebook interactivo, donde aprenderás a construir y desplegar un **agente inteligente con memoria persistente** usando las tecnologías más avanzadas de Databricks:

- **LangGraph**: Orquestación flexible de agentes conversacionales.
- **Lakebase**: Almacenamiento de memoria de conversación respaldado por PostgreSQL.
- **MLflow**: Registro, versionado y despliegue de modelos.
- **Unity Catalog**: Organización y seguridad de recursos.
- **Model Serving**: Despliegue en producción con endpoints de LLM.

---

## 🚀 ¿Qué vas a lograr aquí?

- Crear un agente que **recuerda conversaciones** usando IDs de hilo.
- Optimizar la gestión de conexiones para alta concurrencia.
- Registrar y desplegar el modelo en MLflow y Unity Catalog.
- Probar la memoria del agente y su capacidad de streaming.
- Preparar tu entorno para producción siguiendo las mejores prácticas de Databricks FE.

---

> **Sigue los pasos del notebook para descubrir cómo los agentes con memoria pueden transformar la experiencia conversacional en tu Lakehouse.**

</div>

## ⚙️ Configuración Actual del Notebook

### 📌 Valores que DEBES revisar/cambiar:

 Componente | Ubicación | Valor Actual | ¿Necesita cambio? |
------------|-----------|--------------|--------------------|
 **Lakebase Instance** | Cell 5, línea 42 | `dv-agents-memory` | ✅ Sí, si usas otra instancia |
 **LLM Endpoint** | Cell 5, línea 38 | `databricks-claude-3-7-sonnet` | ✅ Sí, si no está disponible |
 **UC Catalog** | Cell 13 | `users` | ✅ Sí, cambia a tu Catalogo  |
 **UC Schema** | Cell 13 | `daniel_vargas` | ✅ Sí, cambia a tu schema |
 **Model Name** | Cell 13 | `stateful_lakebase_agent` | 🟡 Opcional (pero recomendado) |

---

# Agente LangGraph con memoria persistente en Lakebase

Este notebook demuestra cómo construir y desplegar un **agente con estado** usando:
* **LangGraph** - Para la orquestación del agente
* **Lakebase** - Para memoria persistente de conversación (respaldado por PostgreSQL)
* **MLflow** - Para seguimiento y despliegue del modelo
* **Unity Catalog** - Para el registro de modelos
* **Databricks Model Serving** - Para despliegue en producción

## Características clave
* ✅ **Memoria de conversación**: El agente recuerda interacciones pasadas usando IDs de hilo
* ✅ **Pool de conexiones optimizado**: Maneja alta concurrencia sin timeouts
* ✅ **Listo para producción**: Totalmente rastreado, evaluado y desplegado
* ✅ **Escalable**: Patrón singleton para uso eficiente de recursos

## Arquitectura

Entrada del usuario → Agente LangGraph → LLM (Claude 3.7 Sonnet)
                ↓
        Lakebase (PostgreSQL)
        - Almacena el estado de la conversación
        - Memoria basada en hilos
        - Pool de conexiones

In [0]:
# Instala versiones compatibles de todos los paquetes
print("📦 Instalando dependencias (esto toma 2-3 minutos)...\n")

# Instala paquetes con versiones compatibles
%pip install -U --quiet \
  'databricks-langchain[memory]>=0.16.0' \
  'databricks-ai-bridge>=0.15.0' \
  'databricks-vectorsearch>=0.50' \
  'mlflow>=3.0.0' \
  'langgraph>=1.0.0' \
  'databricks-agents>=0.9.0'

print("\n✅ ¡Instalación completa!")
print("🔄 Reiniciando Python para cargar los nuevos paquetes...\n")

dbutils.library.restartPython()

In [0]:
%%writefile agent.py
import logging
import os
import uuid
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    CheckpointSaver,
)
from databricks.sdk import WorkspaceClient
from langchain_core.messages import AIMessage, AIMessageChunk, AnyMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
)

logger = logging.getLogger(__name__)
logging.basicConfig(level=os.getenv("LOG_LEVEL", "INFO"))

############################################
# Define tu endpoint LLM y prompt del sistema
############################################
LLM_ENDPOINT_NAME = "databricks-claude-3-7-sonnet"
SYSTEM_PROMPT = "Eres un asistente útil. Usa las herramientas disponibles para responder preguntas."

############################################
# Configuración de Lakebase
############################################
LAKEBASE_INSTANCE_NAME = "dv-agents-memory"

# Configuración optimizada del pool de conexiones para psycopg ConnectionPool
LAKEBASE_POOL_CONFIG = {
    "max_size": 50,  # Tamaño máximo del pool (aumentado desde el valor por defecto ~10)
    "min_size": 2,   # Tamaño mínimo del pool
    "timeout": 120.0,  # Tiempo de espera para obtener conexión (2 minutos)
    "max_waiting": 10,  # Permitir hasta 10 solicitudes en cola para conexiones
    "max_lifetime": 1800.0,  # Vida máxima de la conexión (30 minutos)
    "max_idle": 60.0,  # Liberar conexiones inactivas después de 1 minuto
    "reconnect_timeout": 60.0,  # Tiempo para intentar reconexión
}

###############################################################################
## Define herramientas para tu agente
###############################################################################
tools = []

# Ejemplo de herramientas UC; agrega las tuyas según sea necesario
UC_TOOL_NAMES: list[str] = []
if UC_TOOL_NAMES:
    uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
    tools.extend(uc_toolkit.tools)

# Herramientas de búsqueda vectorial
VECTOR_SEARCH_TOOLS = []
tools.extend(VECTOR_SEARCH_TOOLS)

#####################
## Define la lógica del agente
#####################

class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]

# Instancia global de checkpointer para reutilizar conexiones (patrón singleton)
_checkpointer_instance = None

def get_checkpointer():
    """Obtiene o crea una instancia singleton de CheckpointSaver."""
    global _checkpointer_instance
    if _checkpointer_instance is None:
        _checkpointer_instance = CheckpointSaver(
            instance_name=LAKEBASE_INSTANCE_NAME,
            **LAKEBASE_POOL_CONFIG
        )
    return _checkpointer_instance

class LangGraphResponsesAgent(ResponsesAgent):
    """Agente con estado usando ResponsesAgent con checkpointing de Lakebase en pool."""

    def __init__(self, lakebase_config: dict[str, Any]):
        self.workspace_client = WorkspaceClient()
        self.model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
        self.system_prompt = SYSTEM_PROMPT
        self.model_with_tools = self.model.bind_tools(tools) if tools else self.model

    def _create_graph(self, checkpointer: Any):
        def should_continue(state: AgentState):
            messages = state["messages"]
            last_message = messages[-1]
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                return "continue"
            return "end"

        preprocessor = (
            RunnableLambda(lambda state: [{"role": "system", "content": self.system_prompt}] + state["messages"])
            if self.system_prompt
            else RunnableLambda(lambda state: state["messages"])
        )
        model_runnable = preprocessor | self.model_with_tools

        def call_model(state: AgentState, config: RunnableConfig):
            response = model_runnable.invoke(state, config)
            return {"messages": [response]}

        workflow = StateGraph(AgentState)
        workflow.add_node("agente", RunnableLambda(call_model))

        if tools:
            workflow.add_node("herramientas", ToolNode(tools))
            workflow.add_conditional_edges("agente", should_continue, {"continue": "herramientas", "end": END})
            workflow.add_edge("herramientas", "agente")
        else:
            workflow.add_edge("agente", END)

        workflow.set_entry_point("agente")
        return workflow.compile(checkpointer=checkpointer)

    def _get_or_create_thread_id(self, request: ResponsesAgentRequest) -> str:
        """Obtiene thread_id del request o crea uno nuevo.

        Prioridad:
        1. Usa thread_id de custom_inputs si está presente
        2. Usa conversation_id del contexto de chat si está disponible
        3. Genera un nuevo UUID

        Retorna:
            thread_id: El identificador de hilo a usar para esta conversación
        """
        ci = dict(request.custom_inputs or {})

        if "thread_id" in ci:
            return ci["thread_id"]

        # usando conversation id del contexto de chat como thread id
        if request.context and getattr(request.context, "conversation_id", None):
            return request.context.conversation_id

        # Genera nuevo thread_id
        return str(uuid.uuid4())

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        thread_id = self._get_or_create_thread_id(request)
        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        # Convierte mensajes Responses entrantes a formato ChatCompletions
        cc_msgs = self.prep_msgs_for_cc_llm([i.model_dump() for i in request.input])
        langchain_msgs = cc_msgs
        checkpoint_config = {"configurable": {"thread_id": thread_id}}

        # Usa instancia singleton de checkpointer para reutilizar conexiones
        checkpointer = get_checkpointer()
        graph = self._create_graph(checkpointer)

        for event in graph.stream(
            {"messages": langchain_msgs},
            checkpoint_config,
            stream_mode=["updates", "messages"],
        ):
            if event[0] == "updates":
                for node_data in event[1].values():
                    if len(node_data.get("messages", [])) > 0:
                        yield from output_to_responses_items_stream(node_data["messages"])
            elif event[0] == "messages":
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and chunk.content:
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=chunk.content, item_id=chunk.id),
                        )
                except Exception as exc:
                    logger.error("Error transmitiendo chunk: %s", exc)


# ----- Exportar modelo -----
mlflow.langchain.autolog()
AGENTE = LangGraphResponsesAgent(LAKEBASE_INSTANCE_NAME)
mlflow.models.set_model(AGENTE)

In [0]:
from agent import AGENT

print("🧪 Prueba 1: Crear nuevo hilo de conversación\n")

# Primer mensaje - establecer contexto
result1 = AGENT.predict({
    "input": [{"role": "user", "content": "Estoy trabajando en agentes con estado usando Lakebase"}]
})

# Extraer thread_id de la respuesta
thread_id = result1.custom_outputs["thread_id"]
response_text = result1.output[0].content[0]['text']  

print(f"✅ ¡Respuesta recibida!")
print(f"ID de hilo: {thread_id}")
print(f"\nAgente: {response_text[:200]}...")
print(f"\n💾 Estado de la conversación guardado en Lakebase")

In [0]:
print("🧪 Prueba 2: Verificar memoria de la conversación\n")
print(f"Usando thread_id: {thread_id}\n")

# Segundo mensaje - debe recordar el contexto
result2 = AGENT.predict({
    "input": [{"role": "user", "content": "¿En qué estoy trabajando?"}],
    "custom_inputs": {"thread_id": thread_id}
})

response_text = result2.output[0].content[0]['text']  

print(f"✅ ¡Respuesta recibida!")
print(f"\nAgente: {response_text}")

# Verifica si el agente recordó el contexto
if "stateful" in response_text.lower() or "lakebase" in response_text.lower():
    print("\n🎉 ÉXITO - ¡El agente recordó la conversación!")
    print("   La persistencia de memoria vía Lakebase funciona correctamente")
else:
    print("\n⚠️  La respuesta del agente no referencia claramente el contexto previo")
    print("   Es posible que la memoria requiera verificación")

In [0]:
print("🧪 Prueba 3: Probar streaming con memoria\n")
print(f"Usando thread_id: {thread_id}\n")

print("Agente (streaming): ", end="")

for event in AGENT.predict_stream({
    "input": [{"role": "user", "content": "Resume lo que hemos discutido"}],
    "custom_inputs": {"thread_id": thread_id}
}):
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

print("\n\n✅ ¡Prueba de streaming completa!")
print("\n🎯 Todas las pruebas pasaron - El agente está listo para despliegue")

In [0]:
# Reinicia Python para limpiar todas las conexiones de Lakebase de pruebas previas
print("🔄 Reiniciando Python para limpiar el pool de conexiones...\n")
print("Esto asegura que el registro en MLflow tenga acceso limpio a Lakebase\n")

dbutils.library.restartPython()

# Registrar el modelo en MLflow es importante porque permite:

- Guardar y versionar el modelo de manera segura y organizada.
- Rastrear la evolución del modelo a lo largo del tiempo.
- Facilitar la colaboración entre equipos.
- Desplegar el modelo fácilmente en producción.
- Cumplir con políticas de retención y organización de recursos en Databricks.

In [0]:
import mlflow
from mlflow.models.resources import DatabricksLakebase

print("📦 Registrando agente en MLflow...\n")

# Ejemplo de entrada para validación del modelo
input_example = {
    "input": [{"role": "user", "content": "¡Hola! ¿Puedes ayudarme con análisis de datos?"}]
}

# Registra el agente declarando el recurso Lakebase
with mlflow.start_run() as run:
    logged_agent_info = mlflow.pyfunc.log_model(
        python_model="agent.py",
        artifact_path="agent",
        input_example=input_example,
        resources=[DatabricksLakebase(database_instance_name="dv-agents-memory")],  # Corregido: nombre de parámetro correcto
    )

print(f"✅ ¡Agente registrado exitosamente!")
print(f"   ID de ejecución: {run.info.run_id}")
print(f"   URI del modelo: {logged_agent_info.model_uri}")
print(f"   Recurso Lakebase: dv-agents-memory")
print(f"\n🔗 Ver en MLflow: {mlflow.get_experiment(run.info.experiment_id).name}")

## Registrando nuestro agente a Unity Catalog

In [0]:
# Configuración de registro en Unity Catalog

catalog = "users"  
schema = "daniel_vargas"  
model_name = "stateful_lakebase_agent"  # Nombre descriptivo del modelo

print(f"📍 Configuración de registro de modelo en UC:")
print(f"   Catálogo: {catalog}")
print(f"   Esquema: {schema}")
print(f"   Modelo: {model_name}")
print(f"\n   Nombre completo del modelo: {catalog}.{schema}.{model_name}")

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

print(f"📚 Registrando modelo en Unity Catalog...")
print(f"   Destino: {UC_MODEL_NAME}")

# Registra el modelo en UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, 
    name=UC_MODEL_NAME
)

print(f"✅ ¡Modelo registrado exitosamente!")
print(f"   Nombre: {UC_MODEL_NAME}")
print(f"   Versión: {uc_registered_model_info.version}")
print(f"\n🔗 Ver en UC: Catalog Explorer → {catalog} → {schema} → {model_name}")

In [0]:
import mlflow

# Buscar todas las versiones del modelo en Unity Catalog
from mlflow.tracking import MlflowClient
client = MlflowClient()

try:
    # Para Unity Catalog, buscar versiones del modelo
    versions = client.search_model_versions(f"name='{model_name}'")
    
    if versions:
        # Obtener la última versión (número de versión más alto)
        latest_version = max(versions, key=lambda v: int(v.version))
        model_version = latest_version.version
        
        model_info = client.get_model_version(model_name, model_version)
        print(model_info)
    else:
        print(f"No se encontraron versiones para el modelo: {model_name}")
        print("Por favor ejecuta la Celda 11 (Registrar en Unity Catalog) primero.")
except Exception as e:
    print(f"Error: {e}")
    print("Por favor ejecuta la Celda 11 (Registrar en Unity Catalog) primero.")


## 🧪 Probar tu agente en Review Apps de Databricks

Una vez que hayas desplegado tu agente en Model Serving (ver la celda siguiente), puedes probarlo fácilmente usando **Review Apps** de Databricks:

1. **Despliega el agente** usando la celda de abajo ("Deploy Agent").
2. Cuando el despliegue termine, aparecerá un enlace a la Review App en la salida de la celda.
3. Haz clic en el enlace para abrir la Review App. Allí podrás interactuar con tu agente, enviar mensajes y verificar que la memoria persistente funciona correctamente.
4. Puedes probar diferentes hilos de conversación, enviar preguntas y ver cómo el agente recuerda el contexto gracias a Lakebase.

> **Tip:** Si el endpoint aún está iniciando, espera unos minutos y vuelve a cargar la Review App.

---

Continúa con la celda siguiente para desplegar tu agente y obtener el enlace de prueba.

In [0]:
from databricks import agents
from mlflow.tracking import MlflowClient

UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

print(f"🚀 Desplegando agente en Model Serving...")
print(f"   Modelo: {UC_MODEL_NAME}\n")

# Obtener la última versión del modelo
client = MlflowClient()
versions = client.search_model_versions(f"name='{UC_MODEL_NAME}'")

if versions:
    # Obtener la última versión (número de versión más alto)
    latest_version = max(versions, key=lambda v: int(v.version))
    model_version = latest_version.version
    
    print(f"   Versión: {model_version}\n")
    
    # Desplegar el agente
    deployment = agents.deploy(
        UC_MODEL_NAME, 
        model_version, 
        tags={"endpointSource": "stateful-agent-v2"},
        deploy_feedback_model=False
    )
    
    print(f"\n✅ ¡Despliegue iniciado!")
    print(f"\n🕗 Esto puede tomar hasta 15 minutos...")
    print(f"\n🔗 Accede a tu agente:")
    print(f"   Review App: Consulta el resultado arriba para el enlace")
    print(f"   Estado del endpoint: ML → Serving Endpoints → {deployment.endpoint_name if hasattr(deployment, 'endpoint_name') else 'agents_' + UC_MODEL_NAME.replace('.', '-')}")
else:
    print(f"❌ No se encontraron versiones para el modelo: {UC_MODEL_NAME}")
    print("Por favor ejecuta la Celda 11 (Registrar en Unity Catalog) primero.")